# COCO Food Diffusion · Try the trained adapter

Run on a Colab GPU. Download [coco-food-lora.safetensors](https://github.com/MedvAx-AI/coco-food-diffusion/blob/main/weights/coco-food-lora.safetensors) from this repository. Upload it into the notebook when prompted. The demo downloads the pinned Stable Diffusion base model and generates two images from your prompt and seed. It takes no training data or optimizer steps.

In [ ]:
%pip -q install diffusers==0.35.2 transformers==4.56.2 peft==0.17.1 accelerate==1.10.1

In [ ]:
from hashlib import sha256
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from diffusers import DDIMScheduler, StableDiffusionPipeline
from google.colab import files
from peft import LoraConfig, get_peft_model_state_dict, set_peft_model_state_dict
from safetensors.torch import load_file

assert torch.cuda.is_available(), "Choose a GPU in Runtime → Change runtime type."
adapter = Path("/content/coco-food-lora.safetensors")
if not adapter.exists():
    uploaded = files.upload()
    if adapter.name not in uploaded:
        raise ValueError(f"Upload {adapter.name}.")
    adapter.write_bytes(uploaded[adapter.name])
assert adapter.exists(), f"Upload {adapter.name} to /content."
assert sha256(adapter.read_bytes()).hexdigest() == "8bad61aaa803fafb20446c6080b686f97a4a2500e447d70a56e24aa995aaa008"

base_model = "stable-diffusion-v1-5/stable-diffusion-v1-5"
revision = "451f4fe16113bff5a5d2269ed5ad43b0592e9a14"
pipe = StableDiffusionPipeline.from_pretrained(
    base_model, revision=revision, torch_dtype=torch.float16, variant="fp16", use_safetensors=True
).to("cuda")
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe.unet.add_adapter(LoraConfig(
    r=8, lora_alpha=8, target_modules=r".*\.attn2\.(to_q|to_k|to_v|to_out\.0)",
))
state = load_file(str(adapter))
expected = get_peft_model_state_dict(pipe.unet)
assert state.keys() == expected.keys()
assert all(state[key].shape == expected[key].shape for key in state)
loaded = set_peft_model_state_dict(pipe.unet, state)
assert not loaded.unexpected_keys and not any("lora_" in key for key in loaded.missing_keys)

In [ ]:
prompt = "Two pink frosted donuts beside a cup of coffee on a table."  # @param {type:"string"}
seed = 101  # @param {type:"integer"}

def generate():
    noise = torch.randn(
        (1, 4, 48, 48), device="cuda", dtype=torch.float16,
        generator=torch.Generator(device="cuda").manual_seed(seed),
    )
    return pipe(prompt, height=384, width=384, num_inference_steps=24,
                guidance_scale=7.5, latents=noise, eta=0.0).images[0]

pipe.unet.disable_adapters()
pretrained = generate()
pipe.unet.enable_adapters()
tuned = generate()

fig, axes = plt.subplots(1, 2, figsize=(8, 4.5), layout="constrained")
for axis, image, label in zip(axes, [pretrained, tuned], ["Pretrained", "COCO LoRA"]):
    axis.imshow(image)
    axis.set_title(label)
    axis.axis("off")
fig.suptitle(f"{prompt} (seed {seed})")
plt.show()